In [73]:
import os
import pandas as pd
import numpy as np
import re
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Hemolytik 2.0 (new version))

This notebook curates the **Hemolytik 2.0 (new version)** dataset by integrating hemolytic peptide sequences from csv file. The source includes natural peptides, non-natural (chemically modified) variants, and peptide sequences derived from PDB structures. Here we standardize all inputs, separate natural and modified sequences, perform duplicate consistency checks, and export curated datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Toxic measurment:** LD50, LC50, HC50 and MHC
- **Source:** Hemolytik 2.0 (new version)
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads sequence-based source** provided by Hemolytik 2.0 (new version):
  - a comprehensive annotation table (`allsequences.csv`).
- **Separates sequences into natural and modified subsets**:
  - *non-modified*: free N- and C-termini and no non-natural modifications,
  - *modified*: any N-terminal, C-terminal, or non-natural modification.
- **Builds unified datasets**:
  - all sequences are normalized to uppercase,
  - all entries are assigned hemolytic label (`label = 1`), for `Low hemolytic` or `nan`
  - modified sequences retain an explicit `modification` annotation.
- **Checks duplicated sequences** independently for natural and modified datasets:
  - unique sequences are preserved,
  - duplicates with consistent labels are collapsed,
  - conflicting cases (if any) are reported as errors.
- **Extracting the Activities** independently for natural and modified datasets (LD50, LC50, HC50 and MHC)

- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:

 - for classification
  - `processed_hemolytic2.0_new_dataset.csv` (natural sequences),
  - `modified_hemolytic2.0_new_dataset.csv` (modified sequences with annotations),
  - `metadata.json`.
- for regression  
  - `HC50_hemolytic2.0_new_processed_data.csv` (natural sequences),
  - `LC50_hemolytic2.0_new_processed_data.csv` (natural sequences),
  - `LD50_hemolytic2.0_new_processed_data.csv` (natural sequences),
  - `MHC_hemolytic2.0_new_processed_data.csv` (natural sequences),
  - `modified_HC50_hemolytic2.0_new_processed_data.csv`(modified sequences with annotations),
  - `modified_LC50_hemolytic2.0_new_processed_data.csv`(modified sequences with annotations),
  - `modified_LD50_hemolytic2.0_new_processed_data.csv`(modified sequences with annotations),
  - `modified_MHC_hemolytic2.0_new_processed_data.csv`(modified sequences with annotations),
  - `metadata.json`.



In [74]:
name_source = "Hemolytik2.0_new"
name_task = "toxic_effect_classification"
name_task_2 = "toxic_effect_regression"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [75]:
df_data_raw = pd.read_csv(f"{PATH_INPUT}/{name_source}/Hemolytik2_complete_data.csv")
df_data_raw.shape

(13215, 20)

In [76]:
mask_no_mod = (df_data_raw["nter"] == "Free") & (df_data_raw["cter"] == "Free") & (df_data_raw["lyn_cyc"] == "Linear") &(df_data_raw["non_nat"].isna())
df_non_modified = (
    df_data_raw.loc[mask_no_mod, ["seq", "source", "non_hem", "activity"]]
    .reset_index(drop=True)
)

df_non_modified.rename(columns={"seq": "sequence"}, inplace=True)

df_non_modified.shape

(4509, 4)

In [77]:
df_non_modified["non_hem"].unique()

df_non_modified['label'] = df_non_modified['non_hem'].apply(lambda x: 1 if x == "Low hemolytic" or pd.isna(x) else 0)

In [78]:
df_non_modified

,sequence,source,non_hem,activity,label
0,ALWMTLLKKVLKAAAKAALNAVLVGANA,Human,NaN,LC50 = 1.4±0.2 µM,1
1,ALWMTLLKKVLKAAAKAALDAVLVGANA,Human,NaN,LC50 = 1.2±0.1 µM,1
2,ALWDTLLKKVLKAAAKAALNAVLVGANA,Human,NaN,LC50 = 2.3±0.3 µM,1
3,ALWDTLLKKVLKAAAKAALDAVLVGANA,Human,NaN,LC50 = 5±1 µM,1
4,ALWMTLLKKVLKAAAKAALKAVLVGANA,Human,NaN,LC50 = 1.2±0.4 µM,1
...,...,...,...,...,...
4504,GLLGGLLGPLLGGGGGGGGGLL,Human,Non-hemolytic,0 % Hemolytic at 2-200 µM,0
4505,QGIGVGDNDGKRGKR,Human,Low hemolytic,Hemolytic at 160 µM,1
4506,PVVDTTGNNPLQQQEEYYV,Human,Non-hemolytic,0 % Hemolytic at 250 μg/ml,0
4507,MDDSQWVSIHIRDRLAQGNITIRESFLYEGQFHSPEDEKKALTEDD...,Sheep,NaN,Hemolytic at 313 μg/ml,1


In [79]:
df_modified = (
    df_data_raw
    .loc[
        ~mask_no_mod,
        ["seq", "nter", "cter", "non_nat", "source", "non_hem", "activity"]
    ]
    .assign(
        **{
            "non_nat": lambda d: d[
                ["nter", "cter", "non_nat"]
            ].apply(
                lambda row: ";".join(
                    str(x).strip() for x in row
                    if pd.notna(x) and str(x).strip() != ""
                ),
                axis=1
            )
        }
    )
    [["seq", "non_nat", "source", "non_hem", "activity"]]
    .reset_index(drop=True)
)

df_modified.rename(columns={"seq": "sequence"}, inplace=True)

df_modified.shape

(8706, 5)

In [80]:
df_modified['label'] = df_modified['non_hem'].apply(lambda x: 1 if x == "Low hemolytic" or pd.isna(x) else 0)

- Checking duplicates

In [81]:
df_remove_duplicated_nonmodified, df_errors_nonmodified, df_unique_nonmodified = processing_duplicated(df_non_modified, group_seq="sequence", sort_key="label")
df_full_nonmodified = pd.concat([df_unique_nonmodified, df_remove_duplicated_nonmodified], axis=0)

In [82]:
organism_map = (
    df_non_modified
    .drop_duplicates(subset="sequence")
    .set_index("sequence")["source"]
)

activity_map = (
    df_non_modified
    .drop_duplicates(subset="sequence")
    .set_index("sequence")["activity"]
)

df_full_nonmodified["source"] = df_full_nonmodified["sequence"].map(organism_map)
df_full_nonmodified["activity"] = df_full_nonmodified["sequence"].map(activity_map)

df_full_nonmodified.drop(columns=["non_hem"], inplace=True)
df_full_nonmodified

,sequence,source,activity,label
0,wglrrllkygkrs,Human,50 % Hemolysis at >4000 μg/ml,1
1,AAIGSSKPK,Sheep,1.37 % Hemolysis at 500 µM,1
2,AAKIILNPKFRCKAAFC,Horse,little Hemolysis % at 120 µM,1
3,AAKKVLKLLKKLL,Mus musculus,Hemolytic,1
4,AATAKKGAKKADAPAKPKKATKPKSPKKAAKKAGAKKGVKRAGKKG...,Human,0 % Hemolysis at 100 μg/ml,0
...,...,...,...,...
712,CGLFEAIEGFIENPWEGMIDGWYGYGRKKRRQRR,Human,50 % Hemolysis at 1.71 µM (pH 5.5),1
713,GFGSFLGKALKAGLKLGANLLGGAPQQ,Human,LC50 = 145 µM,1
714,GIWGTLAKIGIKAVPRVISMLKKKKQ,Horse,zone of inhibition (1 mm) at 0.4-0.5mM,1
715,CGLFEAIEGFIENNWEGMIDGWYGYGRKKRRQRR,Human,50 % Hemolysis at 6.39 µM (pH 5.5),1


In [83]:
df_errors_nonmodified.shape

(98, 1)

In [84]:
df_remove_duplicated_mod, df_errors_mod, df_unique_mod = processing_duplicated(df_modified, group_seq="sequence", sort_key="label")
df_full_mod = pd.concat([df_unique_mod, df_remove_duplicated_mod], axis=0)

In [85]:
organism_map = (
    df_modified
    .drop_duplicates(subset="sequence")
    .set_index("sequence")["source"]
)

activity_map = (
    df_modified
    .drop_duplicates(subset="sequence")
    .set_index("sequence")["activity"]
)

df_full_mod["source"] = df_full_mod["sequence"].map(organism_map)
df_full_mod["activity"] = df_full_mod["sequence"].map(activity_map)

df_full_mod.drop(columns=["non_hem"], inplace=True)
df_full_mod

,sequence,non_nat,source,activity,label
0,GIVKOIVKOIVKOI,Free;Amidation;O = Ornithine,Human,50 % Hemolysis at 4000 µM,1
1,GIVKKIVKKIVKKI,Free;Amidation,Human,50 % Hemolysis at 1500 µM,1
2,GIRKWFKKAAHVGKKVGKVALNAYL,Free;Amidation,Human,50% hemolytic at >256 μg/ml,1
3,GIPCGESCVYIPCITAALGCSCKSKVCYRN,Free;Free,Human,50 % Hemolysis at 11.1 ± 0.02 µM,1
4,GIPCGESCVWIPCISSAIGCSCKSKVCYRN,Free;Free,Human,HD50 % Hemolytic at 36 µM,1
...,...,...,...,...,...
1371,lklksivswakkvl,NaN,Guinea-pig,82-85% Hemolysis at 15 µM,1
1372,kﬀkrllksvrravkkfrk,NaN,Mouse,31 % Hemolysis at 50 µM,1
1373,yPyP,NaN,Human,100% Hemolysis at >150 μg/ml,1
1374,(Cip)-CFQWKRAMRKVR,NaN,Human,0 % Hemolysis at 200 μg/ml,0


In [86]:
df_errors_mod.shape

(185, 1)

- Extracting the Activities

In [87]:
df_full_nonmodified["sequence"] = (
        df_full_nonmodified["sequence"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

In [88]:
df_full_nonmodified_regr_raw = df_full_nonmodified[
    df_full_nonmodified["activity"].str.contains(
        r"MHC|LD50|HC50|LC50",
        case=False,
        na=False
    )
]

In [89]:
df_full_mod_regr_raw = df_full_mod[
    df_full_mod["activity"].str.contains(
        r"MHC|LD50|HC50|LC50",
        case=False,
        na=False
    )
]

In [90]:
def parse_activities(df):

    df_parsed = df.copy()

    # ============================================================
    # 1. Limpiar activity
    # ============================================================

    df_parsed["activity_type"] = (
        df_parsed["activity"]
        .astype("string")
        .str.strip()
    )


    # ============================================================
    # 2. Identificar endpoint
    # ============================================================

    df_parsed["activity_type_clean"] = (
        df_parsed["activity_type"]
        .str.extract(
            r"(MHC|LD50|HC50|LC50)",
            expand=False
        )
    )


    # ============================================================
    # 3. Extraer relación
    # ============================================================

    def extract_relation(text):

        if pd.isna(text):
            return np.nan

        text = str(text)

        # Normalizar símbolos Unicode
        text = (
            text
            .replace("≥", ">=")
            .replace("≤", "<=")
            .replace("−", "-")
        )

        # Quitar el endpoint para no confundir el "50"
        # de LD50/HC50/LC50 con el valor
        text_without_endpoint = re.sub(
            r"^(?:MHC|LD50|HC50|LC50)",
            "",
            text,
            count=1
        )

        # Buscar operadores
        matches = list(
            re.finditer(
                r">=|<=|>|<|=",
                text_without_endpoint
            )
        )

        # Buscar primer número después del endpoint
        number_match = re.search(
            r"\d+(?:[.,]\d+)?",
            text_without_endpoint
        )

        if not number_match:
            return np.nan

        number_pos = number_match.start()

        # Operadores antes del número
        previous = [
            m for m in matches
            if m.start() < number_pos
        ]

        if previous:
            return previous[-1].group()

        # Si no hay operador, asumimos =
        return "="


    df_parsed["activity_relation"] = (
        df_parsed["activity_type"]
        .apply(extract_relation)
    )


    # ============================================================
    # 4. Extraer valor
    # ============================================================

    df_parsed["activity_value"] = (
        df_parsed["activity_type"]
        .str.extract(
            r"(?:MHC|LD50|HC50|LC50).*?"
            r"(\d+(?:[.,]\d+)?"
            r"(?:\s*(?:\*|x|×)\s*10\s*[-−+]?\s*\d+)?)",
            expand=False
        )
    )


    # ============================================================
    # 5. Convertir valor
    # ============================================================

    def parse_number(value):

        if pd.isna(value):
            return np.nan

        value = str(value)

        # Normalizar
        value = value.replace("−", "-")
        value = value.replace("×", "x")
        value = value.replace(",", ".")
        value = value.replace(" ", "")

        # Notación científica
        if "x10" in value:

            base, exponent = value.split("x10")

            return float(base) * (10 ** int(exponent))

        elif "*10" in value:

            base, exponent = value.split("*10")

            return float(base) * (10 ** int(exponent))

        # Número normal
        return float(value)


    df_parsed["activity_value"] = (
        df_parsed["activity_value"]
        .apply(parse_number)
        .astype(float)
    )


    # ============================================================
    # 6. Extraer error ±
    # ============================================================

    error_pm = (
        df_parsed["activity_type"]
        .str.extract(
            r"±\s*(\d+(?:[.,]\d+)?)",
            expand=False
        )
        .astype("string")
        .str.replace(",", ".", regex=False)
    )

    error_pm = pd.to_numeric(
        error_pm,
        errors="coerce"
    )


    # ============================================================
    # 7. Extraer error [x]
    # ============================================================

    error_bracket = (
        df_parsed["activity_type"]
        .str.extract(
            r"\[(\d+(?:[.,]\d+)?)\]",
            expand=False
        )
        .astype("string")
        .str.replace(",", ".", regex=False)
    )

    error_bracket = pd.to_numeric(
        error_bracket,
        errors="coerce"
    )


    # ============================================================
    # 8. Combinar errores
    #
    # Primero usa ±
    # Si no existe, usa [x]
    # ============================================================

    df_parsed["activity_error"] = (
        error_pm
        .fillna(error_bracket)
        .astype(float)
    )


    # ============================================================
    # 9. Extraer unidad
    # ============================================================

    df_parsed["activity_unit"] = (
        df_parsed["activity_type"]
        .str.extract(
            r"(µM|μM|uM|"
            r"µg/ml|μg/ml|ug/ml|"
            r"mg/L|mg/l|"
            r"μmol/L|µmol/L|"
            r"mM|"
            r"g\s*/\s*ml|"
            r"g\s+ml-1|"
            r"ml-1)",
            expand=False
        )
    )


    # ============================================================
    # 10. Normalizar unidades
    # ============================================================

    df_parsed["activity_unit"] = (
        df_parsed["activity_unit"]
        .astype("string")
        .str.replace(r"\s+", "", regex=True)
        .replace({
            "μM": "µM",
            "uM": "µM",

            "μmol/L": "µM",
            "µmol/L": "µM",

            "μg/ml": "μg/mL",
            "ug/ml": "μg/mL",

            "mg/l": "mg/L",

            "gml-1": "g/mL",
            "g/ml": "g/mL"
        })
    )


    return df_parsed

In [91]:
df_full_nonmodified_regres = parse_activities(df_full_nonmodified_regr_raw)

In [92]:
df_full_mod_regres = parse_activities(df_full_mod_regr_raw)

In [93]:
df_full_mod_regres[
    [
        "activity",
        "activity_type_clean",
        "activity_value",
        "activity_error",
        "activity_relation",
        "activity_unit"
    ]
].head(20)

,activity,activity_type_clean,activity_value,activity_error,activity_relation,activity_unit
63,LC50 = 30 µM,LC50,30.0,NaN,=,µM
64,LC50 = 10 µM,LC50,10.0,NaN,=,µM
68,MHC <10 μg/ml,MHC,10.0,NaN,<,μg/mL
73,LD50 = 350 µM,LD50,350.0,NaN,=,µM
74,LD50 = 80 µM,LD50,80.0,NaN,=,µM
75,LD50 = 110 µM,LD50,110.0,NaN,=,µM
76,LD50 = 60 µM,LD50,60.0,NaN,=,µM
77,LD50 = 15 µM,LD50,15.0,NaN,=,µM
78,LD50 = 25 µM,LD50,25.0,NaN,=,µM
82,LD50 >400 µM,LD50,400.0,NaN,>,µM


In [94]:
df_full_nonmodified_regres["activity_unit"].unique()

<ArrowStringArray>
['µM', 'μg/mL']
Length: 2, dtype: string

In [95]:
# Ver las filas donde value quedó como NaN

df_full_mod_regres.loc[df_full_mod_regres["activity"].isna()]

,sequence,non_nat,source,activity,label,activity_type,activity_type_clean,activity_relation,activity_value,activity_error,activity_unit


In [96]:
def create_label(row):

    relation = row["activity_relation"]
    value = row["activity_value"]
    error = row["activity_error"]

    if pd.isna(value):
        return np.nan

    value_str = f"{value:g}"

    # Valor exacto
    if relation == "=":
        if not pd.isna(error):
            return f"{value_str} ± {error:g}"
        return value_str

    # Valor con límite
    if not pd.isna(error):
        return f"{relation} {value_str} ± {error:g}"

    return f"{relation} {value_str}"


df_full_nonmodified_regres["label"] = df_full_nonmodified_regres.apply(
    create_label,
    axis=1
)

df_full_mod_regres["label"] = df_full_mod_regres.apply(
    create_label,
    axis=1
)


In [97]:
df_full_nonmodified_regres = df_full_nonmodified_regres[["sequence", 'source', "label", "activity_unit", "activity_type_clean"]]
df_full_nonmodified_regres.rename(columns= {"activity_unit": "unit"}, inplace=True)
df_full_nonmodified_regres

,sequence,source,label,unit,activity_type_clean
29,GIFSKINKKKAKTGLFNIIKTVGKEAGMDVIRAGIDTISCKIKGEC,Human,180,µM,HC50
31,GIFPKIIGKGIVNGIKSLAKGVGMKVFKAGLNNIGNTGCNNRDEC,Human,200,µM,HC50
40,GFWTTAAEGLKKFAKAGLASILNPK,Human,105,µM,LC50
47,GFMKYIKPLIPHAVKAISKLI,Human,135,µM,LD50
48,GFMKYIKPLIPHAVKAIKKLI,Human,200,µM,LD50
...,...,...,...,...,...
678,ALWKTLLKKVLKAAAK,Human,57 ± 3,µM,LC50
700,GIMDSVKGLAKNLAGKLLDSLKCKITGC,Human,> 200,µM,HC50
701,GILSTFKGLAKGVAKDLAGNLLDKFKCKITGC,Human,120,µM,HC50
703,GFLGPLLKLAAKGVAKVIPHLIPSRQQ,Human,90,µM,HC50


In [98]:
df_full_mod_regres

,sequence,non_nat,source,activity,label,activity_type,activity_type_clean,activity_relation,activity_value,activity_error,activity_unit
63,GLKKIFKAGLGSLVKGIK*AHVAS,Free;Free;* = palmitate(PAL),Human,LC50 = 30 µM,30,LC50 = 30 µM,LC50,=,30.0,NaN,µM
64,GLKEIFKAGLGSLVKGIAAHVAS,palmitate(pal);Free,Human,LC50 = 10 µM,10,LC50 = 10 µM,LC50,=,10.0,NaN,µM
68,GLGSVFGRLARIGRVIPKV,Free;Amidation,Human,MHC <10 μg/ml,< 10,MHC <10 μg/ml,MHC,<,10.0,NaN,μg/mL
73,GLLGPLLKIAKKVGSNLL,Free;Amidation,Human,LD50 = 350 µM,350,LD50 = 350 µM,LD50,=,350.0,NaN,µM
74,GLLGPLLKIAAKVGSKLL,Free;Amidation,Human,LD50 = 80 µM,80,LD50 = 80 µM,LD50,=,80.0,NaN,µM
...,...,...,...,...,...,...,...,...,...,...,...
1356,akklohalhoallalohlahollakk,NaN,Human,HC50 = 716.8 µM,716.8,HC50 = 716.8 µM,HC50,=,716.8,NaN,µM
1359,fPVOLfPVOL,NaN,Human,HC50 = 17.5±1.07 µM,17.5 ± 1.07,HC50 = 17.5±1.07 µM,HC50,=,17.5,1.07,µM
1361,kWkSFLkTFkSLKkTVLHTLLkAISS,NaN,Human,MHC >325.20μmol/L,> 325.2,MHC >325.20μmol/L,MHC,>,325.2,NaN,µM
1367,kwksflktfksavktvlhtalkaiss,NaN,Human,HC50 = 1.8 µM,1.8,HC50 = 1.8 µM,HC50,=,1.8,NaN,µM


In [99]:
df_full_mod_regres = df_full_mod_regres[["sequence", 'non_nat', 'source', "label", "activity_unit", "activity_type_clean"]]
df_full_mod_regres.rename(columns= {"activity_unit": "unit"}, inplace=True)
df_full_mod_regres

,sequence,non_nat,source,label,unit,activity_type_clean
63,GLKKIFKAGLGSLVKGIK*AHVAS,Free;Free;* = palmitate(PAL),Human,30,µM,LC50
64,GLKEIFKAGLGSLVKGIAAHVAS,palmitate(pal);Free,Human,10,µM,LC50
68,GLGSVFGRLARIGRVIPKV,Free;Amidation,Human,< 10,μg/mL,MHC
73,GLLGPLLKIAKKVGSNLL,Free;Amidation,Human,350,µM,LD50
74,GLLGPLLKIAAKVGSKLL,Free;Amidation,Human,80,µM,LD50
...,...,...,...,...,...,...
1356,akklohalhoallalohlahollakk,NaN,Human,716.8,µM,HC50
1359,fPVOLfPVOL,NaN,Human,17.5 ± 1.07,µM,HC50
1361,kWkSFLkTFkSLKkTVLHTLLkAISS,NaN,Human,> 325.2,µM,MHC
1367,kwksflktfksavktvlhtalkaiss,NaN,Human,1.8,µM,HC50


- Parsing the activities

In [100]:
df_lc50 = df_full_nonmodified_regres[
    df_full_nonmodified_regres["activity_type_clean"] == "LC50"
].copy()

df_lc50.drop(columns=["activity_type_clean"], inplace=True)

df_hc50 = df_full_nonmodified_regres[
    df_full_nonmodified_regres["activity_type_clean"] == "HC50"
].copy()

df_hc50.drop(columns=["activity_type_clean"], inplace=True)


df_ld50 = df_full_nonmodified_regres[
    df_full_nonmodified_regres["activity_type_clean"] == "LD50"
].copy()

df_ld50.drop(columns=["activity_type_clean"], inplace=True)


df_mhc = df_full_nonmodified_regres[
    df_full_nonmodified_regres["activity_type_clean"] == "MHC"
].copy()

df_mhc.drop(columns=["activity_type_clean"], inplace=True)

In [101]:
df_lc50_mod = df_full_mod_regres[
    df_full_mod_regres["activity_type_clean"] == "LC50"
].copy()

df_lc50_mod.drop(columns=["activity_type_clean"], inplace=True)

df_hc50_mod = df_full_mod_regres[
    df_full_mod_regres["activity_type_clean"] == "HC50"
].copy()

df_hc50_mod.drop(columns=["activity_type_clean"], inplace=True)


df_ld50_mod = df_full_mod_regres[
    df_full_mod_regres["activity_type_clean"] == "LD50"
].copy()

df_ld50_mod.drop(columns=["activity_type_clean"], inplace=True)


df_mhc_mod = df_full_mod_regres[
    df_full_mod_regres["activity_type_clean"] == "MHC"
].copy()

df_mhc_mod.drop(columns=["activity_type_clean"], inplace=True)

- Working with metada

In [102]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [103]:
dict_metadata.update({
    "number_of_raw_sequences": df_data_raw.shape[0],
    "number_of_sequences_retained": len(df_full_nonmodified),
    "number_of_positive_sequences": int((df_full_nonmodified["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full_nonmodified["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors_nonmodified),
    "number_of_modified_sequences" : len(df_full_mod),
    "number_of_erroneous_modified_sequences" : len(df_errors_mod),
    "modified_sequences_included": False,
    "number_of_LC50": df_lc50.shape[0],
    "number_of_HC50": df_hc50.shape[0],
    "number_of_LD50": df_ld50.shape[0],
    "number_of_MHC": df_mhc.shape[0],
    "number_of_LC50_mod": df_lc50_mod.shape[0],
    "number_of_HC50_mod": df_hc50_mod.shape[0],
    "number_of_LD50_mod": df_ld50_mod.shape[0],
    "number_of_MHC_mod": df_mhc_mod.shape[0]
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2026,
 'last update date': datetime.datetime(2026, 5, 14, 0, 0),
 'download date': Timestamp('2026-09-02 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative, HC50, LD50, LC50, MHC',
 'unit of measurement': 'µM, mg/L',
 'obtaining negative dataset': 'Experimentally validated',
 'repository or server': 'https://github.com/raghavagps/Hemolytik2/tree/Database',
 'publication': 'https://pubmed.ncbi.nlm.nih.gov/41525503/',
 'number_of_raw_sequences': 13215,
 'number_of_sequences_retained': 2487,
 'number_of_positive_sequences': 1966,
 'number_of_negative_sequences': 521,
 'number_of_erroneous_sequences': 98,
 'number_of_modified_sequences': 4270,
 'number_of_erroneous_modified_sequences': 185,
 'modified_sequences_included': False,
 'number_of_LC50': 56,
 'number_of_HC50': 43,
 'number_of_LD50': 21,
 'number_of_MHC'

- Exporting data

In [104]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [105]:
os.makedirs(f"{PATH_EXPORT}/{name_task_2}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task_2}/{name_source}/metadata.json", dict_metadata)

In [106]:
df_full_nonmodified.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytik2.0_new_dataset.csv", index=False)
df_full_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytik2.0_new_dataset.csv", index=False)

In [107]:
df_lc50.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/LC50_hemolytik2.0_new_processed_data.csv", index=False)
df_ld50.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/LD50_hemolytik2.0_new_processed_data.csv", index=False)
df_hc50.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/HC50_hemolytik2.0_new_processed_data.csv", index=False)
df_mhc.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/MHC_hemolytik2.0_new_processed_data.csv", index=False)

In [108]:
df_lc50_mod.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/modified_LC50_hemolytik2.0_new_processed_data.csv", index=False)
df_ld50_mod.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/modified_LD50_hemolytik2.0_new_processed_data.csv", index=False)
df_hc50_mod.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/modified_HC50_hemolytik2.0_new_processed_data.csv", index=False)
df_mhc_mod.to_csv(f"{PATH_EXPORT}/{name_task_2}/{name_source}/modified_MHC_hemolytik2.0_new_processed_data.csv", index=False)